# PRISM Rebuttal - regularisation sensitivity in the OOD experiments

Reviewer fWEj asked:

> What is the sensitivity of the regularization parameter C **in the OOD
> calibration experiment**? An ablation study here will be helpful to understand
> its effect.

The earlier sweep covered the in-distribution setting only. This one covers the
transfer setting the question was about: all four pairs, all eight models,
C over five values, three label fractions, three seeds.

**1,440 logistic-regression fits.** CPU only, roughly 30 to 60 minutes.
Checkpointed per (model, pair).

What the analysis reports:

1. Spread in OOD AUROC and OOD ECE across C, so the two can be compared.
2. Whether the *ranking* of models by OOD calibration is stable under C, which
   is the question that matters for a benchmark.
3. Whether the reverse-scaling direction on MHIST to PCam is preserved at every
   value of C, which is the question that matters for the headline claim.

In [1]:
import os, gc, glob, time, warnings
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from scipy.stats import kendalltau
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/PRISM'
EMB_DIR = f'{BASE}/embeddings'
OUT_DIR = f'{BASE}/results_v2'
CKPT    = f'{OUT_DIR}/ood_c_parts'
os.makedirs(CKPT, exist_ok=True)

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
MKEYS  = ['clip','plip','conch','virchow2','uni','gigapath','h_optimus_0','midnight']
M2K    = dict(zip(MODELS, MKEYS))

D2K = {'PCam':'pcam', 'MHIST':'mhist', 'CRC':'crc', 'BRACS':'bracs'}
OOD_PAIRS = [('MHIST','PCam'), ('PCam','MHIST'), ('CRC','BRACS'), ('BRACS','CRC')]

C_GRID    = [0.01, 0.1, 1.0, 10.0, 100.0]
FRACTIONS = [0.01, 0.10, 1.00]
SEEDS     = [42, 123, 456]
N_BINS, MAX_ITER = 15, 1000

CRC_BIN   = lambda y: (y == 8).astype(int)
BRACS_BIN = lambda y: np.isin(y, [1, 3]).astype(int)

print('grid:', len(MODELS), 'models x', len(OOD_PAIRS), 'pairs x',
      len(C_GRID), 'C x', len(FRACTIONS), 'fractions x', len(SEEDS), 'seeds =',
      len(MODELS)*len(OOD_PAIRS)*len(C_GRID)*len(FRACTIONS)*len(SEEDS), 'fits')

Mounted at /content/drive
grid: 8 models x 4 pairs x 5 C x 3 fractions x 3 seeds = 1440 fits


## 1. Shared helpers, identical to the corrected transfer run

In [2]:
def _ece_edges(conf, correct, edges):
    ece, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum() > 0:
            ece += m.sum() * abs(correct[m].mean() - conf[m].mean())
    return float(ece / n)

def conf_correct(proba, y):
    return proba[:, 1], (y == 1).astype(float)

def ece_fixed(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    return _ece_edges(c, k, np.linspace(0, 1, n_bins + 1))

def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(val_logits, val_y, bounds=(0.1, 10.0)):
    idx = np.arange(len(val_y))
    def nll(T):
        p = softmax_np(val_logits / T)
        return float(-np.log(p[idx, val_y] + 1e-12).mean())
    return float(minimize_scalar(nll, bounds=bounds, method='bounded').x)

def stratified_sample(labels, fraction, seed):
    np.random.seed(seed)
    idx_all = np.arange(len(labels))
    picked = []
    for c in np.unique(labels):
        ci = idx_all[labels == c]
        picked.extend(np.random.choice(ci, size=max(1, int(len(ci)*fraction)),
                                       replace=False))
    return np.array(sorted(picked))

def degeneracy(pred):
    cnt = np.bincount(pred, minlength=2)
    return float(cnt.max() / cnt.sum())

def load_emb(mkey, dkey, split):
    p = f'{EMB_DIR}/{mkey}/{dkey}'
    return (np.load(f'{p}/{split}_features.npy', mmap_mode='r'),
            np.load(f'{p}/{split}_labels.npy').astype(int))

def load_side(mk, name):
    dk = D2K[name]
    Xtr, ytr = load_emb(mk, dk, 'train')
    Xte, yte = load_emb(mk, dk, 'test')
    try:
        Xva, yva = load_emb(mk, dk, 'val')
    except FileNotFoundError:
        Xva, yva = None, None
    if name == 'CRC':
        ytr, yte = CRC_BIN(ytr), CRC_BIN(yte)
        yva = None if yva is None else CRC_BIN(yva)
    elif name == 'BRACS':
        ytr, yte = BRACS_BIN(ytr), BRACS_BIN(yte)
        yva = None if yva is None else BRACS_BIN(yva)
    return Xtr, ytr, Xva, yva, Xte, yte

def proba_chunked(clf, X, chunk=20000):
    return np.vstack([clf.predict_proba(np.asarray(X[i:i+chunk], dtype=np.float32))
                      for i in range(0, X.shape[0], chunk)])

def logits_chunked(clf, X, chunk=20000):
    out = []
    for i in range(0, X.shape[0], chunk):
        d = clf.decision_function(np.asarray(X[i:i+chunk], dtype=np.float32))
        if d.ndim == 1:
            d = d.reshape(-1, 1); d = np.hstack([-d, d])
        out.append(d)
    return np.vstack(out)

print('ready')

ready


## 2. Sweep C on every transfer pair

In [3]:
def run_pair_C(model_name, src, tgt):
    mk = M2K[model_name]
    Xs_tr, ys_tr, Xs_va, ys_va, _, _ = load_side(mk, src)
    _, _, _, _, Xt_te, yt_te = load_side(mk, tgt)
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx = stratified_sample(ys_tr, frac, seed)
            Xs = np.asarray(Xs_tr[idx], dtype=np.float32)
            ys = ys_tr[idx]
            for C in C_GRID:
                clf = LogisticRegression(max_iter=MAX_ITER, C=C,
                                         random_state=seed).fit(Xs, ys)
                proba = proba_chunked(clf, Xt_te)
                pred  = proba.argmax(1)
                try:
                    auroc = roc_auc_score(yt_te, proba[:, 1])
                except Exception:
                    auroc = np.nan

                if Xs_va is not None:
                    T  = fit_temperature(logits_chunked(clf, Xs_va), ys_va)
                    sp = softmax_np(logits_chunked(clf, Xt_te) / T)
                    ece_s = ece_fixed(sp, yt_te)
                    del sp
                else:
                    T = ece_s = np.nan

                rows.append(dict(
                    model=model_name, src=src, tgt=tgt, pair=f'{src}->{tgt}',
                    fraction=frac, seed=seed, C=C, n_train=len(idx),
                    auroc=auroc,
                    f1_macro=f1_score(yt_te, pred, average='macro',
                                      zero_division=0),
                    brier=brier_score_loss(yt_te, proba[:, 1]),
                    ece_fixed=ece_fixed(proba, yt_te),
                    temperature_src=T, ece_scaled_fixed=ece_s,
                    degeneracy_share=degeneracy(pred)))
                del proba, clf
            del Xs
            gc.collect()

    del Xs_tr, Xs_va, Xt_te
    gc.collect()
    return pd.DataFrame(rows)


t0 = time.time()
for src, tgt in OOD_PAIRS:
    for model_name in MODELS:
        out = f'{CKPT}/{M2K[model_name]}__{D2K[src]}_to_{D2K[tgt]}.csv'
        if os.path.exists(out):
            print(f'  skip: {model_name} {src}->{tgt}')
            continue
        try:
            df = run_pair_C(model_name, src, tgt)
            df.to_csv(out, index=False)
            sp_a = (df.groupby(['fraction','seed'])['auroc']
                      .agg(lambda v: v.max()-v.min()).mean())
            sp_e = (df.groupby(['fraction','seed'])['ece_fixed']
                      .agg(lambda v: v.max()-v.min()).mean())
            print(f'{model_name:>12} {src:>6}->{tgt:<6} '
                  f'C-spread auroc={sp_a:.4f} ece={sp_e:.4f}  '
                  f'({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'{model_name:>12} {src:>6}->{tgt:<6} FAILED: '
                  f'{type(e).__name__}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT}/*.csv'))
df_c = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df_c.to_csv(f'{OUT_DIR}/ood_c_ablation.csv', index=False)
print(f'\n{len(parts)}/32 pairs, {len(df_c)} fits -> ood_c_ablation.csv')

        CLIP  MHIST->PCam   C-spread auroc=0.0939 ece=0.1751  (30s)
        PLIP  MHIST->PCam   C-spread auroc=0.0914 ece=0.1673  (63s)
       CONCH  MHIST->PCam   C-spread auroc=0.0460 ece=0.2080  (88s)
    VIRCHOW2  MHIST->PCam   C-spread auroc=0.1121 ece=0.2280  (140s)
         UNI  MHIST->PCam   C-spread auroc=0.1250 ece=0.1903  (176s)
    GigaPath  MHIST->PCam   C-spread auroc=0.0964 ece=0.2489  (216s)
 H-Optimus-0  MHIST->PCam   C-spread auroc=0.0536 ece=0.2301  (253s)
    MIDNIGHT  MHIST->PCam   C-spread auroc=0.0819 ece=0.2115  (294s)
        CLIP   PCam->MHIST  C-spread auroc=0.0521 ece=0.1311  (492s)
        PLIP   PCam->MHIST  C-spread auroc=0.1531 ece=0.1785  (709s)
       CONCH   PCam->MHIST  C-spread auroc=0.0423 ece=0.1760  (857s)
    VIRCHOW2   PCam->MHIST  C-spread auroc=0.0354 ece=0.1717  (1205s)
         UNI   PCam->MHIST  C-spread auroc=0.0210 ece=0.1937  (1326s)
    GigaPath   PCam->MHIST  C-spread auroc=0.1069 ece=0.2101  (1477s)
 H-Optimus-0   PCam->MHIST  C-spre

## 3. How much does C move the OOD metrics?

The comparison that answers the question: spread in OOD AUROC against spread in
OOD ECE, averaged over models and seeds.

In [4]:
spread = lambda v: v.max() - v.min()

print('=== Spread across C, averaged over models and seeds ===\n')
for metric in ['auroc', 'ece_fixed', 'ece_scaled_fixed']:
    s = (df_c.groupby(['pair','fraction','model','seed'])[metric]
             .agg(spread).groupby(level=[0,1]).mean())
    print(f'--- {metric} ---')
    print(s.unstack().round(4).to_string())
    print()

a = (df_c.groupby(['pair','fraction','model','seed'])['auroc']
         .agg(spread).groupby(level=[0,1]).mean())
e = (df_c.groupby(['pair','fraction','model','seed'])['ece_fixed']
         .agg(spread).groupby(level=[0,1]).mean())
r = (e / a).rename('ece_over_auroc')
print('=== ECE spread divided by AUROC spread ===')
print('values above 1 mean calibration is more C-sensitive than discrimination')
print(r.unstack().round(2).to_string())

=== Spread across C, averaged over models and seeds ===

--- auroc ---
fraction       0.01    0.10    1.00
pair                               
BRACS->CRC   0.1274  0.1964  0.1880
CRC->BRACS   0.0612  0.0686  0.0619
MHIST->PCam  0.0420  0.0789  0.1418
PCam->MHIST  0.0542  0.0774  0.0622

--- ece_fixed ---
fraction       0.01    0.10    1.00
pair                               
BRACS->CRC   0.1239  0.1218  0.0885
CRC->BRACS   0.1565  0.1397  0.1007
MHIST->PCam  0.1452  0.2264  0.2505
PCam->MHIST  0.2333  0.2020  0.1241

--- ece_scaled_fixed ---
fraction       0.01    0.10    1.00
pair                               
BRACS->CRC   0.1192  0.1772  0.1549
CRC->BRACS   0.1436  0.0843  0.0512
MHIST->PCam  0.1116  0.2394  0.2601
PCam->MHIST  0.0964  0.0932  0.0754

=== ECE spread divided by AUROC spread ===
values above 1 mean calibration is more C-sensitive than discrimination
fraction     0.01  0.10  1.00
pair                         
BRACS->CRC   0.97  0.62  0.47
CRC->BRACS   2.56  2.04  1.62


## 4. Is the model ranking stable under C?

For a benchmark this is the question that matters: if changing C reorders the
models, then any calibration-based ranking is protocol-dependent.

In [5]:
print('Kendall tau against C = 1.0, per (pair, fraction)\n')
rows = []
for pair in df_c['pair'].unique():
    for frac in FRACTIONS:
        base = (df_c[(df_c.pair==pair) & (df_c.fraction==frac) & (df_c.C==1.0)]
                .groupby('model')[['auroc','ece_fixed']].mean())
        for C in C_GRID:
            if C == 1.0:
                continue
            alt = (df_c[(df_c.pair==pair) & (df_c.fraction==frac) & (df_c.C==C)]
                   .groupby('model')[['auroc','ece_fixed']].mean())
            k = base.index.intersection(alt.index)
            if len(k) < 3:
                continue
            rows.append(dict(pair=pair, fraction=frac, C=C,
                tau_auroc=kendalltau(base.loc[k,'auroc'],
                                     alt.loc[k,'auroc']).correlation,
                tau_ece=kendalltau(base.loc[k,'ece_fixed'],
                                   alt.loc[k,'ece_fixed']).correlation))

df_rank = pd.DataFrame(rows)
df_rank.to_csv(f'{OUT_DIR}/ood_c_ranking.csv', index=False)

print(df_rank.groupby(['pair','fraction'])[['tau_auroc','tau_ece']]
      .mean().round(3).to_string())

print('\n=== Overall ===')
n = len(df_rank)
print(f'  comparisons              : {n}')
print(f'  mean tau, AUROC ranking  : {df_rank.tau_auroc.mean():.3f}')
print(f'  mean tau, ECE ranking    : {df_rank.tau_ece.mean():.3f}')
print(f'  sign inversions, AUROC   : {(df_rank.tau_auroc < 0).sum()} of {n}')
print(f'  sign inversions, ECE     : {(df_rank.tau_ece < 0).sum()} of {n}')

Kendall tau against C = 1.0, per (pair, fraction)

                      tau_auroc  tau_ece
pair        fraction                    
BRACS->CRC  0.01          0.750    0.482
            0.10          0.589    0.357
            1.00          0.429    0.357
CRC->BRACS  0.01          0.732    0.679
            0.10          0.679    0.839
            1.00          0.768    0.768
MHIST->PCam 0.01          0.875    0.536
            0.10          0.786    0.804
            1.00          0.696    0.661
PCam->MHIST 0.01          0.732    0.393
            0.10          0.786    0.661
            1.00          0.714    0.589

=== Overall ===
  comparisons              : 48
  mean tau, AUROC ranking  : 0.711
  mean tau, ECE ranking    : 0.594
  sign inversions, AUROC   : 0 of 48
  sign inversions, ECE     : 2 of 48


## 5. Does the headline claim survive every value of C?

The reverse-scaling claim covers UNI, VIRCHOW2, GigaPath and H-Optimus-0 on
MHIST to PCam. If it holds at all five values of C, it is not an artefact of the
regularisation choice.

In [6]:
CLAIM = ['UNI','VIRCHOW2','GigaPath','H-Optimus-0']
sub = df_c[df_c.pair == 'MHIST->PCam']

print('MHIST->PCam, raw ECE at 1% and 100% source labels, per C\n')
hdr = f"{'model':>12} {'C':>7} {'ECE@1%':>9} {'ECE@100%':>10} {'rises':>7}"
print(hdr); print('-'*len(hdr))
ok = {}
for m in CLAIM:
    for C in C_GRID:
        s = sub[(sub.model == m) & (sub.C == C)].groupby('fraction')['ece_fixed'].mean()
        if len(s) < 2:
            continue
        lo, hi = s.loc[0.01], s.loc[1.00]
        rises = hi > lo
        ok[(m, C)] = rises
        print(f'{m:>12} {C:>7} {lo:>9.3f} {hi:>10.3f} {"YES" if rises else "no":>7}')
    print()

n_ok = sum(ok.values())
print(f'=== The claim holds in {n_ok} of {len(ok)} (model, C) combinations ===')
if n_ok == len(ok):
    print('    Holds at every value of C for all four models.')
else:
    fails = [f'{m} at C={C}' for (m, C), v in ok.items() if not v]
    print('    Exceptions:', ', '.join(fails))

print('\n=== Degeneracy across C on MHIST->PCam ===')
print(sub.pivot_table(index='model', columns='C',
                      values='degeneracy_share').round(3).to_string())

MHIST->PCam, raw ECE at 1% and 100% source labels, per C

       model       C    ECE@1%   ECE@100%   rises
-------------------------------------------------
         UNI    0.01     0.222      0.221      no
         UNI     0.1     0.222      0.284     YES
         UNI     1.0     0.225      0.439     YES
         UNI    10.0     0.262      0.478     YES
         UNI   100.0     0.291      0.443     YES

    VIRCHOW2    0.01     0.222      0.246     YES
    VIRCHOW2     0.1     0.222      0.359     YES
    VIRCHOW2     1.0     0.231      0.453     YES
    VIRCHOW2    10.0     0.295      0.486     YES
    VIRCHOW2   100.0     0.329      0.487     YES

    GigaPath    0.01     0.222      0.233     YES
    GigaPath     0.1     0.224      0.326     YES
    GigaPath     1.0     0.244      0.462     YES
    GigaPath    10.0     0.334      0.497     YES
    GigaPath   100.0     0.426      0.498     YES

 H-Optimus-0    0.01     0.222      0.232     YES
 H-Optimus-0     0.1     0.223      0.3

## 6. In-distribution against transfer

Places the new numbers next to the earlier in-distribution sweep so the two can
be quoted together.

In [7]:
try:
    ind_c = pd.read_csv(f'{OUT_DIR}/c_ablation.csv')
    a_i = (ind_c.groupby(['dataset','fraction','model'])['auroc']
                .agg(spread).groupby(level=[1]).mean())
    e_i = (ind_c.groupby(['dataset','fraction','model'])['ece']
                .agg(spread).groupby(level=[1]).mean())
    a_o = (df_c.groupby(['pair','fraction','model','seed'])['auroc']
               .agg(spread).groupby(level=[1]).mean())
    e_o = (df_c.groupby(['pair','fraction','model','seed'])['ece_fixed']
               .agg(spread).groupby(level=[1]).mean())
    comp = pd.DataFrame({'auroc_in': a_i, 'ece_in': e_i,
                         'auroc_ood': a_o, 'ece_ood': e_o})
    comp['ratio_in']  = comp.ece_in  / comp.auroc_in
    comp['ratio_ood'] = comp.ece_ood / comp.auroc_ood
    print('Mean spread across C, by label fraction\n')
    print(comp.round(4).to_string())
    print('\nratio above 1: calibration more C-sensitive than discrimination')
except FileNotFoundError:
    print('c_ablation.csv not found; run the in-distribution sweep first')

Mean spread across C, by label fraction

          auroc_in  ece_in  auroc_ood  ece_ood  ratio_in  ratio_ood
fraction                                                           
0.01        0.0222  0.1041     0.0712   0.1647    4.6805     2.3146
0.10        0.0609  0.1290     0.1053   0.1725    2.1191     1.6378
1.00        0.0891  0.1641     0.1135   0.1409    1.8414     1.2422

ratio above 1: calibration more C-sensitive than discrimination


## 7. What to report

The answer to the reviewer's question is whatever the tables above say. Three
outcomes and how each should be written up.

**C barely moves OOD metrics and the ranking is stable.** Report the spreads and
state that the transfer conclusions are insensitive to the regularisation
choice.

**C moves OOD ECE much more than OOD AUROC, and reorders the calibration
ranking.** This is what the in-distribution sweep found, and if it reproduces
here it is the stronger result: it means calibration-based comparison requires a
fixed probe protocol, in transfer as well as in distribution.

**The headline claim fails at some value of C.** Then it is bounded to the
regularisation setting used and the paper must say so. Report which values and
which models.